# XGB v3 — OOT Feature Redundancy & Correlation Analysis

**Goal:** determine whether the important OOT permutation features provide unique information or are mostly redundant.

This notebook is diagnostic only. It does **not** drop features or retrain the model.

Expected existing variables:
- `oot_sample`
- `top_features`
- `config`
- optionally `results` from OOT permutation importance


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


## 1. Select the features from permutation importance

NameError: name 'top_features' is not defined

## 2. Pull only those features from the OOT Spark dataframe

## 3. Separate numeric and categorical features

## 4. Numeric correlation — Spearman

Use Spearman because several mortgage variables are skewed or count/bounded variables.

Rough diagnostic interpretation:
- `< 0.50` — low redundancy
- `0.50–0.80` — investigate
- `0.80–0.90` — high redundancy
- `> 0.90` — very high redundancy

These are **diagnostic thresholds**, not automatic feature-dropping rules.


## 5. Strongest numeric feature pairs

## 6. Delinquency feature family

Pay particular attention to:

- `max_dpd_6m`
- `max_dpd_12m`
- `delinquency_months_6m`
- `delinquency_months_12m`
- `months_since_last_delinquency`

The question is whether these are distinct signals or different representations of the same delinquency history.


## 7. Visualize the strongest numeric relationships

## 8. Categorical overlap — Cramér's V

Pearson/Spearman correlation is not appropriate for nominal variables.

Cramér's V:
- `0` ≈ little association
- `1` = perfect association


## 9. Mixed numeric/categorical overlap — correlation ratio

This checks whether a categorical feature is effectively acting as a proxy for a numeric feature.


## 10. Combine permutation importance with redundancy

Interpretation:

- **High permutation importance + low correlation** → strong unique candidate
- **High permutation importance + high correlation** → important but overlapping
- **Low permutation importance + high correlation** → potentially replaceable
- **Low permutation importance + low correlation** → weak individual contribution

Do not automatically remove correlated features. XGBoost can use correlated features in different tree splits/interactions.


# 11. What to do with the results

### If an important feature has low redundancy
Keep it. It is stronger evidence of unique predictive information.

### If several important features are highly correlated
Do **not** immediately delete them.

Instead, run a targeted ablation experiment:
1. remove one feature/family;
2. retrain the XGB model;
3. evaluate on the exact same OOT population;
4. compare ROC-AUC, PR-AUC and calibration.

### If gain is high but permutation importance is low
The feature may be redundant with another feature or mainly useful through interactions.

### If permutation importance is materially negative
Investigate it with ablation. Tiny negative values can simply be sampling variation.

## The key question

> **Does removing the redundant feature actually hurt OOT performance?**

That is the next experiment after this notebook.
